# 06 · k-means and interests

Maths unit **M8**. Read `rec/r5_interests.py`.

R5 replaces one taste vector with up to eight, found by k-means on the liked images, and scores by a soft maximum over them. This notebook runs the M8 examples, finds the interests of a simulated six-interest user, and shows Hungarian matching keeping interest ids stable when the clusters are recomputed.

**Everything below that involves a user is SIMULATED** (`eval/sim.py`). Real labels only exist once Alen swipes.

In [1]:
import glob, json, time
import numpy as np
import matplotlib.pyplot as plt

from visualtaste.api.deps import AppState
from visualtaste.config import get_settings

state = AppState(get_settings())
state.load()
features = state.require_features()
print(f"{len(features):,} pool images loaded")

Archive ladder reached rule 5: no external drive and only 42.7 GB free. Using the internal fallback by prior decision (docs/DECISIONS.md #2).


50,000 pool images loaded


## 1 · M8 by hand

In [2]:
from visualtaste.rec.r5_interests import lloyd, logsumexp_score

print("1-D k-means from (1, 2):", lloyd(np.array([1.0, 2, 9, 10]), np.array([1.0, 2])).ravel())
for tau in (0.05, 1.0):
    print(f"log-sum-exp of (0.40, 0.10, 0.05) at tau = {tau}: {logsumexp_score(np.array([0.40, 0.10, 0.05]), tau):.3f}")

1-D k-means from (1, 2): [1.5 9.5]
log-sum-exp of (0.40, 0.10, 0.05) at tau = 0.05: 0.400
log-sum-exp of (0.40, 0.10, 0.05) at tau = 1.0: 1.294


## 2 · Finding six planted interests

An S6 user (six SigLIP interests) is shown 3,000 random images; the tracker clusters what it liked. Each found interest is matched with the planted one it sits closest to.

In [3]:
from visualtaste.eval.sim import make_variant_user
from visualtaste.rec.r5_interests import InterestTracker
from visualtaste.api.deps import concept_labeller

user = make_variant_user("S6_six_interests", features, seed=1729, cluster_labels=features.cluster64)
shown = np.random.default_rng(1729).choice(len(features), 3_000, replace=False)
actions = user.react_rows(features, shown)
liked = features.siglip[shown[np.isin(actions, ["like", "save"])]]
tracker = InterestTracker(seed=1729)
tracker.update(liked, concept_labeller=concept_labeller(state.settings))
state_ = tracker.state
match = state_.centroids @ user.centroids.T
print(f"{len(liked)} likes -> {state_.k} interests")
for j in range(state_.k):
    print(f"  #{state_.ids[j]}: {state_.counts[j]:>4} likes, nearest planted interest {match[j].argmax()} (cos {match[j].max():.2f}) -- {state_.labels[j] if state_.labels else ''}")
print("planted interests recovered:", len(set(match.argmax(axis=1))), "of", user.interest_count)

1365 likes -> 8 interests
  #0:  148 likes, nearest planted interest 2 (cos 0.85) -- landscape, overexposed, hill
  #1:  199 likes, nearest planted interest 2 (cos 0.96) -- shallow depth of field, close-up, overexposed
  #2:  141 likes, nearest planted interest 0 (cos 0.88) -- sport, athlete, performance
  #3:  211 likes, nearest planted interest 5 (cos 0.94) -- overexposed, gallery, cross-processed
  #4:  152 likes, nearest planted interest 4 (cos 0.91) -- lithograph, print, engraving
  #5:   90 likes, nearest planted interest 1 (cos 0.95) -- portrait, lithograph, self-portrait
  #6:  247 likes, nearest planted interest 0 (cos 0.97) -- candid, performance, ceremony
  #7:  177 likes, nearest planted interest 3 (cos 0.94) -- black and white, desaturated, monochrome
planted interests recovered: 6 of 6


## 3 · Reclustering without renumbering

Shuffle the order of the liked images and recluster from scratch. k-means numbers its clusters arbitrarily; Hungarian matching on cosine distance gives each new cluster its old id.

In [4]:
from visualtaste.rec.r5_interests import cluster_likes, match_ids

again = cluster_likes(liked[np.random.default_rng(7).permutation(len(liked))], state_.k, seed=99)
ids, born, retired, _ = match_ids(state_.centroids, state_.ids, again, state_.next_id)
print("new cluster order -> inherited ids:", ids)
print("born:", born, " retired:", retired)
print("each new centroid's cosine to the old one with its id:",
      np.round([float(again[i] @ state_.centroids[state_.ids.index(ids[i])]) if ids[i] in state_.ids else float('nan') for i in range(len(ids))], 3))

new cluster order -> inherited ids: [3, 5, 7, 6, 4, 2, 1, 0]
born: []  retired: []
each new centroid's cosine to the old one with its id: [0.999 0.981 0.996 1.    0.994 1.    0.999 0.999]
